### Import libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset


### Define model architecture

In [ ]:
class KneeClassifier(nn.Module):

    def __init__(
        self,
        feature_dim=768,
        num_classes=12
    ):
        super().__init__()

        self.slice_attention = SliceAttention(
            feature_dim
        )

        self.slot_attention = SlotAttention(
            feature_dim
        )

        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.2),
            
            nn.Linear(256, num_classes)
        )

    def forward(
        self,
        features,
        slice_mask,
        slot_mask
    ):

        # -----------------------------------------
        # Slice attention
        # -----------------------------------------

        slot_features, slice_weights = \
            self.slice_attention(
                features,
                slice_mask
            )

        # [B,6,768]

        # -----------------------------------------
        # Slot attention
        # -----------------------------------------

        study_feature, slot_weights = \
            self.slot_attention(
                slot_features,
                slot_mask
            )

        # [B,768]

        # -----------------------------------------
        # Classification
        # -----------------------------------------

        logits = self.classifier(
            study_feature
        )

        # [B,12]

        return (
            logits,
            slice_weights,
            slot_weights
        )

In [ ]:
class DinoKneeClassifier(nn.Module):

    def __init__(
        self,
        feature_dim=768,
        num_classes=12
    ):
        super().__init__()

        self.slice_attention = SliceAttention(
            feature_dim
        )

        self.slot_attention = SlotAttention(
            feature_dim
        )

        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(256, num_classes)
        )

    def forward(
        self,
        features,
        slice_mask,
        slot_mask
    ):

        # -----------------------------------------
        # Slice attention
        # -----------------------------------------

        slot_features, slice_weights = \
            self.slice_attention(
                features,
                slice_mask
            )

        # [B,6,768]

        # -----------------------------------------
        # Slot attention
        # -----------------------------------------

        study_feature, slot_weights = \
            self.slot_attention(
                slot_features,
                slot_mask
            )

        # [B,768]

        # -----------------------------------------
        # Classification
        # -----------------------------------------

        logits = self.classifier(
            study_feature
        )

        # [B,12]

        return (
            logits,
            slice_weights,
            slot_weights
        )

In [ ]:
class SliceAttention(nn.Module):

    def __init__(self, feature_dim=768):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.Tanh(),
            nn.Linear(256, 1)
        )

    def forward(
        self,
        x,
        slice_mask=None
    ):
        """
        x:
            [B, 6, 9, 768]

        slice_mask:
            [B, 6, 9]
            1 = valid slice
            0 = empty/missing slice
        """

        # -----------------------------------------
        # 1. Calculate attention scores
        # -----------------------------------------
        scores = self.attention(x).squeeze(-1)

        # [B, 6, 9]

        if slice_mask is not None:

            # Do not allow empty slices to participate
            scores = scores.masked_fill(
                slice_mask == 0,
                -1e9
            )

        # -----------------------------------------
        # 2. Softmax over the 9 slices
        # -----------------------------------------
        weights = F.softmax(
            scores,
            dim=2
        )

        # -----------------------------------------
        # 3. Explicitly set empty slices to 0
        # -----------------------------------------
        if slice_mask is not None:

            weights = weights * slice_mask

        # [B, 6, 9]

        # -----------------------------------------
        # 4. Weighted sum of slice features
        # -----------------------------------------
        slot_features = torch.sum(
            x * weights.unsqueeze(-1),
            dim=2
        )

        # [B, 6, 768]

        return slot_features, weights

In [ ]:
class SlotAttention(nn.Module):

    def __init__(self, feature_dim=768):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.Tanh(),
            nn.Linear(256, 1)
        )

    def forward(
        self,
        x,
        slot_mask=None
    ):
        """
        x:
            [B, 6, 768]

        slot_mask:
            [B, 6]
            1 = valid slot
            0 = missing slot
        """

        # -----------------------------------------
        # 1. Calculate attention scores
        # -----------------------------------------
        scores = self.attention(x).squeeze(-1)

        # [B, 6]

        if slot_mask is not None:

            # Prevent missing slots from participating
            scores = scores.masked_fill(
                slot_mask == 0,
                -1e9
            )

        # -----------------------------------------
        # 2. Softmax across the 6 slots
        # -----------------------------------------
        weights = F.softmax(
            scores,
            dim=1
        )

        # -----------------------------------------
        # 3. Explicitly set missing slots to 0
        # -----------------------------------------
        if slot_mask is not None:

            weights = weights * slot_mask

        # [B, 6]

        # -----------------------------------------
        # 4. Weighted combination of slots
        # -----------------------------------------
        study_feature = torch.sum(
            x * weights.unsqueeze(-1),
            dim=1
        )

        # [B, 768]

        return study_feature, weights

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
N_SLICES = 9

### Load coatnet model

In [ ]:
coatnet_model = KneeClassifier().to(device)

coatnet_checkpoint = torch.load(
    "/kaggle/input/models/basith007/coatnet/pytorch/default/1/coatnet_model.pt",
    map_location=device,
    weights_only=False
)

coatnet_model.load_state_dict(coatnet_checkpoint["model_state_dict"])
coatnet_model.to(device)
coatnet_model.eval()

### Load Dino model

In [ ]:
dino_model = DinoKneeClassifier().to(device)

dino_checkpoint = torch.load(
    "/kaggle/input/models/ganeshan26/dino-model/pytorch/default/1/best_knee_model_V2.pt",
    map_location=device,
    weights_only=False
)

dino_model.load_state_dict(dino_checkpoint["model_state_dict"])
dino_model.to(device)
dino_model.eval()

In [ ]:
import pandas as pd

LABEL_PATH = "/kaggle/input/datasets/ganeshan26/train-label/train_label.csv"

df = pd.read_csv(LABEL_PATH)

print(df.shape)
print(df.columns)

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))

In [ ]:
import os
import glob
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Update these two paths to your mounted feature datasets.
DINO_FEATURE_DIR = "/kaggle/input/datasets/ganeshan26/dicom-features"
COATNET_FEATURE_DIR = "/kaggle/input/datasets/basith007/coatnet-features"

LABEL_COLS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

# Load labels
train_df = pd.read_csv(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv"
)

print("Labels:", train_df.shape)
print("DINO feature files:", len(glob.glob(
    os.path.join(DINO_FEATURE_DIR, "*.pt")
)))
print("CoAtNet feature files:", len(glob.glob(
    os.path.join(COATNET_FEATURE_DIR, "*.pt")
)))

In [ ]:
# import os
# import glob
# import torch

# def inspect_batched_features(feature_dir, model_name):
#     files = sorted(glob.glob(os.path.join(feature_dir, "batch_*.pt")))

#     total_studies = 0
#     sample_study = None

#     print(f"\n{'='*55}")
#     print(f"{model_name}: {len(files)} batch files")

#     for path in files:
#         batch = torch.load(
#             path, map_location="cpu", weights_only=False
#         )

#         if not isinstance(batch, dict):
#             print("Unexpected batch type:", type(batch))
#             continue

#         total_studies += len(batch)

#         if sample_study is None and batch:
#             study_id = next(iter(batch))
#             sample_study = batch[study_id]
#             print("Sample batch:", os.path.basename(path))
#             print("Sample study ID:", study_id)
#             print("Study data type:", type(sample_study).__name__)

#             if isinstance(sample_study, dict):
#                 print("Study keys:", list(sample_study.keys()))

#                 for key, value in sample_study.items():
#                     if torch.is_tensor(value):
#                         print(f"  {key}: shape={tuple(value.shape)}, dtype={value.dtype}")
#                     else:
#                         print(f"  {key}: {type(value).__name__}")

#     print("Total studies:", total_studies)

# inspect_batched_features(DINO_FEATURE_DIR, "DINOv2")
# inspect_batched_features(COATNET_FEATURE_DIR, "CoAtNet")

In [ ]:
# import os
# import glob
# import torch
# import numpy as np
# import pandas as pd

# # Load the ground-truth labels
# LABEL_COLS = [
#     "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
#     "Medial OA", "Lateral OA", "PF OA", "Effusion",
#     "Synovitis", "Baker's", "Contusion", "Fracture"
# ]

# train_df = pd.read_csv(
#     "/kaggle/input/datasets/ganeshan26/train-label/train_label.csv"
# )

# def load_feature_batches(feature_dir):
#     feature_map = {}

#     files = sorted(glob.glob(os.path.join(feature_dir, "*.pt")))
#     print(f"Loading {len(files)} batch files from {feature_dir}")

#     for path in files:
#         batch = torch.load(
#             path,
#             map_location="cpu",
#             weights_only=False
#         )

#         for study_id, study_data in batch.items():
#             study_id = str(study_id)
#             if study_id in feature_map:
#                 raise ValueError(f"Duplicate study ID: {study_id}")

#             feature_map[study_id] = study_data

#     return feature_map


# dino_features = load_feature_batches(DINO_FEATURE_DIR)
# coatnet_features = load_feature_batches(COATNET_FEATURE_DIR)

# dino_ids = set(dino_features)
# coatnet_ids = set(coatnet_features)
# label_ids = set(train_df["StudyInstanceUID"].astype(str))

# print("\nDINO studies:", len(dino_ids))
# print("CoAtNet studies:", len(coatnet_ids))
# print("Label studies:", len(label_ids))
# print("Common feature studies:", len(dino_ids & coatnet_ids))
# print("Studies missing labels:", len((dino_ids & coatnet_ids) - label_ids))
# print("Studies missing DINO features:", len(label_ids - dino_ids))
# print("Studies missing CoAtNet features:", len(label_ids - coatnet_ids))

# # Use only studies present in both feature datasets and the label CSV.
# common_ids = sorted(dino_ids & coatnet_ids & label_ids)

# assert len(common_ids) > 0, "No matching studies found."

# labels_df = (
#     train_df.assign(
#         StudyInstanceUID=train_df["StudyInstanceUID"].astype(str)
#     )
#     .set_index("StudyInstanceUID")
# )

# y_true = labels_df.loc[common_ids, LABEL_COLS].to_numpy(
#     dtype=np.float32
# )

# print("\nMatched studies:", len(common_ids))
# print("Ground-truth shape:", y_true.shape)

In [ ]:
# from torch.utils.data import Dataset, DataLoader

# class MatchedFeatureDataset(Dataset):
#     def __init__(self, study_ids, feature_map):
#         self.study_ids = study_ids
#         self.feature_map = feature_map

#     def __len__(self):
#         return len(self.study_ids)

#     def __getitem__(self, idx):
#         study_id = self.study_ids[idx]
#         item = self.feature_map[study_id]

#         return (
#             item["features"].float(),
#             item["slice_mask"].float(),
#             item["slot_mask"].float()
#         )


# def predict_model(model, feature_map, study_ids, batch_size=32):
#     dataset = MatchedFeatureDataset(study_ids, feature_map)

#     loader = DataLoader(
#         dataset,
#         batch_size=batch_size,
#         shuffle=False,
#         num_workers=2,
#         pin_memory=(device.type == "cuda")
#     )

#     model.eval()
#     predictions = []

#     with torch.inference_mode():
#         for features, slice_mask, slot_mask in loader:
#             features = features.to(device, non_blocking=True)
#             slice_mask = slice_mask.to(device, non_blocking=True)
#             slot_mask = slot_mask.to(device, non_blocking=True)

#             logits, _, _ = model(
#                 features,
#                 slice_mask,
#                 slot_mask
#             )

#             probs = torch.sigmoid(logits)
#             predictions.append(probs.cpu().numpy())

#     return np.concatenate(predictions, axis=0)


# dino_model.eval()
# coatnet_model.eval()

# dino_probs = predict_model(
#     dino_model, dino_features, common_ids
# )

# coatnet_probs = predict_model(
#     coatnet_model, coatnet_features, common_ids
# )

# # Equal-weight probability ensemble
# ensemble_probs = 0.3 * dino_probs + 0.7 * coatnet_probs

# print("DINO predictions:", dino_probs.shape)
# print("CoAtNet predictions:", coatnet_probs.shape)
# print("Ensemble predictions:", ensemble_probs.shape)

# assert dino_probs.shape == y_true.shape
# assert coatnet_probs.shape == y_true.shape
# assert ensemble_probs.shape == y_true.shape

In [ ]:
# dino_probs

In [ ]:
# from sklearn.metrics import roc_auc_score, average_precision_score

# def evaluate_predictions(y_true, probabilities, model_name):
#     rows = []

#     for i, label in enumerate(LABEL_COLS):
#         y = y_true[:, i]
#         p = probabilities[:, i]

#         if np.unique(y).size < 2:
#             auc = np.nan
#             ap = np.nan
#         else:
#             auc = roc_auc_score(y, p)
#             ap = average_precision_score(y, p)

#         rows.append({
#             "Label": label,
#             "ROC-AUC": auc,
#             "Average Precision": ap
#         })

#     result = pd.DataFrame(rows)

#     print(f"\n{model_name}")
#     print("Mean ROC-AUC:", round(result["ROC-AUC"].mean(), 4))
#     print("Mean Average Precision:", round(
#         result["Average Precision"].mean(), 4
#     ))

#     return result


# dino_results = evaluate_predictions(
#     y_true, dino_probs, "DINOv2"
# )

# coatnet_results = evaluate_predictions(
#     y_true, coatnet_probs, "CoAtNet"
# )

# ensemble_results = evaluate_predictions(
#     y_true, ensemble_probs, "DINOv2 + CoAtNet Ensemble"
# )

# comparison = pd.DataFrame({
#     "Label": LABEL_COLS,
#     "DINO ROC-AUC": dino_results["ROC-AUC"],
#     "CoAtNet ROC-AUC": coatnet_results["ROC-AUC"],
#     "Ensemble ROC-AUC": ensemble_results["ROC-AUC"],
#     "DINO AP": dino_results["Average Precision"],
#     "CoAtNet AP": coatnet_results["Average Precision"],
#     "Ensemble AP": ensemble_results["Average Precision"],
# })

# display(comparison.round(4))

In [ ]:
import os
import glob
import torch
import numpy as np
import pandas as pd
import pydicom

from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, AutoModel

import timm
from timm.data import resolve_model_data_config, create_transform

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

COMP_ROOT = (
    "/kaggle/input/competitions/"
    "rsna-knee-abnormality-detection"
)

TEST_SERIES_ROOT = os.path.join(COMP_ROOT, "test_series")

# Use the map generated in the previous step.
TEST_MAP_PATH = "/kaggle/input/competitions/rsna-knee-abnormality-detection/test_series.csv"

SLOT_COLUMNS = [
    "Axial_F0",
    "Axial_F1",
    "Coronal_F0",
    "Coronal_F1",
    "Sagittal_F0",
    "Sagittal_F1",
]

N_SLOTS = 6
N_SLICES = 9
FEATURE_DIM = 768

test_series_map = pd.read_csv(TEST_MAP_PATH)

test_series_map["StudyInstanceUID"] = (
    test_series_map["StudyInstanceUID"].astype(str)
)

print("Test studies:", len(test_series_map))
display(test_series_map)

### Load dino model for feature extraction

In [ ]:
import torch
from transformers import AutoImageProcessor, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DINO_MODEL_PATH = "/kaggle/input/models/metaresearch/dinov2/pytorch/base/1"

dino_processor = AutoImageProcessor.from_pretrained(
    DINO_MODEL_PATH,
    local_files_only=True
)

dino_encoder = AutoModel.from_pretrained(
    DINO_MODEL_PATH,
    local_files_only=True
).to(device)

dino_encoder.eval()

print("DINOv2 encoder loaded successfully from local Kaggle files.")
print("Device:", device)

### Load coatnet model for feature extraction

In [ ]:
import os
import copy
import torch
import timm

# --------------------------------------------------
# Configuration
# --------------------------------------------------
WEIGHTS_PATH = (
    "/kaggle/input/models/ganeshan26/coatnet-pretrained/"
    "pytorch/default/1/coatnet_0_rw_224_pretrained.pth"
)

USE_FP16 = False
NUM_WORKERS = min(4, os.cpu_count() or 1)

torch.backends.cudnn.benchmark = True

N_GPUS = torch.cuda.device_count()
devices = (
    [torch.device(f"cuda:{i}") for i in range(N_GPUS)]
    if N_GPUS > 0
    else [torch.device("cpu")]
)

print("Devices:", devices)
assert os.path.isfile(WEIGHTS_PATH), f"Weights not found: {WEIGHTS_PATH}"


base_model = timm.create_model(
    "coatnet_0_rw_224",
    pretrained=False,
    num_classes=0,
    global_pool="avg"
)

state_dict = torch.load(
    WEIGHTS_PATH,
    map_location="cpu",
    weights_only=True
)

base_model.load_state_dict(state_dict, strict=True)
print("Pretrained CoAtNet weights loaded.")

models = []

for d in devices:
    m = copy.deepcopy(base_model).to(d)

    if d.type == "cuda":
        m = m.to(memory_format=torch.channels_last)

    m.eval()
    models.append(m)

# Keep the variable names used by your earlier notebook
device = devices[0]
coatnet = models[0]

data_config = timm.data.resolve_model_data_config(coatnet)

coatnet_transform = timm.data.create_transform(
    **data_config,
    is_training=False
)

print(f"CoAtNet loaded successfully on {len(models)} device(s).")

### Normalize image

In [ ]:
def normalize_image(image):
    """
    Convert DICOM pixel array to uint8 [0,255].
    """

    image = image.astype(np.float32)

    min_val = image.min()
    max_val = image.max()

    if max_val == min_val:
        return np.zeros_like(
            image,
            dtype=np.uint8
        )

    image = (
        (image - min_val)
        / (max_val - min_val)
    )

    image = image * 255.0

    image = np.clip(
        image,
        0,
        255
    ).astype(np.uint8)

    return image

In [ ]:
def normalize_stack(stack):
    """
    stack: float/int array [H, W, 3]  ->  uint8 [H, W, 3] in [0, 255]
    """
    stack = stack.astype(np.float32)
 
    min_val = stack.min()
    max_val = stack.max()
 
    if max_val == min_val:
        return np.zeros_like(stack, dtype=np.uint8)
 
    stack = (stack - min_val) / (max_val - min_val) * 255.0
 
    return np.clip(stack, 0, 255).astype(np.uint8)

In [ ]:
def extract_coatnet_features(rgb_images, batch_size=N_SLICES):
    """
    rgb_images: list of uint8 arrays [H, W, 3]
                (channels = previous, current, next slice)
    Returns: tensor [len(rgb_images), 768]
    """
    pil_images = [Image.fromarray(img) for img in rgb_images]
 
    all_features = []
 
    for i in range(0, len(pil_images), batch_size):
 
        batch_images = pil_images[i:i + batch_size]
 
        batch_tensor = torch.stack([
            coatnet_transform(image) for image in batch_images
        ]).to(device)
 
        with torch.no_grad():
            batch_features = coatnet(batch_tensor)
 
        all_features.append(batch_features.cpu())
 
    return torch.cat(all_features, dim=0)

In [ ]:
def extract_dinov2_features(images, batch_size=9):
    """
    Extract DINOv2 features for multiple images.

    Input:
        images:
            List of NumPy uint8 images or PIL Images

        batch_size:
            Number of images processed together.

    Output:
        torch.Tensor [N, 768]
    """

    processed_images = []

    # ---------------------------------------------
    # Convert images to PIL RGB
    # ---------------------------------------------

    for image in images:

        if isinstance(image, np.ndarray):
            image = Image.fromarray(image)

        if image.mode != "RGB":
            image = image.convert("RGB")

        processed_images.append(image)

    # ---------------------------------------------
    # Batched DINOv2 inference
    # ---------------------------------------------

    all_features = []

    for i in range(0, len(processed_images), batch_size):

        batch_images = processed_images[
            i:i + batch_size
        ]

        inputs = dino_processor(
            images=batch_images,
            return_tensors="pt"
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.no_grad():
            outputs = dino_encoder(**inputs)

        # CLS token
        batch_features = outputs.last_hidden_state[:, 0, :]

        all_features.append(
            batch_features.cpu()
        )

    # ---------------------------------------------
    # Combine batches
    # ---------------------------------------------

    features = torch.cat(
        all_features,
        dim=0
    )

    return features

In [ ]:
def get_sorted_dicom_files(series_folder_path):
    """
    Return DICOM files sorted according to their true slice position.

    Uses:
        ImageOrientationPatient
        ImagePositionPatient

    rather than relying on InstanceNumber.

    Args:
        series_folder_path (str):
            Path to a single DICOM series folder.

    Returns:
        list[str]:
            Sorted DICOM file paths.
    """

    # 1. Find DICOM files

    dcm_files = [
        os.path.join(series_folder_path, filename)
        for filename in os.listdir(series_folder_path)
        if filename.lower().endswith(".dcm")
    ]

    if not dcm_files:
        raise ValueError(
            f"No DICOM files found in: {series_folder_path}"
        )

    # 2. Read geometry information]

    slice_info = []

    for filepath in dcm_files:

        ds = pydicom.dcmread(
            filepath,
            stop_before_pixels=True
        )

        # Image position of this slice
        position = np.asarray(
            ds.ImagePositionPatient,
            dtype=np.float64
        )

        # Image orientation:
        # first 3  -> row direction
        # next 3   -> column direction
        orientation = np.asarray(
            ds.ImageOrientationPatient,
            dtype=np.float64
        )

        row_direction = orientation[:3]
        column_direction = orientation[3:]
        
        # Calculate slice normal---

        slice_normal = np.cross(
            row_direction,
            column_direction
        )

        # Calculate position along slice direction

        slice_position = np.dot(
            position,
            slice_normal
        )

        slice_info.append(
            (
                filepath,
                slice_position
            )
        )


    # 3. Sort by actual slice position

    slice_info.sort(
        key=lambda x: x[1]
    )

    # 4. Return only file paths

    sorted_files = [
        filepath
        for filepath, position in slice_info
    ]

    return sorted_files

In [ ]:
def select_9_slices(dicom_files, n_slices=9):
    """
    Select 9 evenly spaced slices from a series.
    """

    n = len(dicom_files)

    if n < n_slices:
        raise ValueError(
            f"Series contains only {n} slices, "
            f"but {n_slices} are required."
        )

    indices = np.linspace(0, n - 1,n_slices,dtype=int)

    return [dicom_files[i] for i in indices]

In [ ]:
def select_indices(n, n_slices=N_SLICES, edge_fraction=0.05):
    """
    Pick `n_slices` evenly spaced indices from the central part of a series.
    If the series is shorter than n_slices, indices repeat (no crash).
    """
    start = int(round(n * edge_fraction))
    end = n - 1 - start

    if end - start + 1 < n_slices:
        start, end = 0, n - 1

    return np.linspace(start, end, n_slices).round().astype(int)

In [ ]:
def extract_coatnet_series_features(series_path, n_slices=N_SLICES, batch_size=N_SLICES):
 
    # 1. Sorted files of the ENTIRE series
    sorted_files = get_sorted_dicom_files(series_path)
    n = len(sorted_files)
 
    print(f"Found {n} slices")
 
    # 2. Which 9 slices do we want in the end?
    indices = select_indices(n, n_slices)
 
    print(f"Selected indices: {indices.tolist()}")
 
    # 3. Read pixel data from the full series (cached; edges clamped)
    cache = {}
 
    def load(idx):
        idx = min(max(idx, 0), n - 1)      # clamp at series edges
        if idx not in cache:
            cache[idx] = pydicom.dcmread(sorted_files[idx]).pixel_array
        return cache[idx]
 
    # 4. Neighbor stacking on the FULL series: [i-1, i, i+1]
    #    then keep only the chosen 9
    rgb_images = []
 
    for i in indices:
        stack = np.stack(
            [load(i - 1), load(i), load(i + 1)],
            axis=-1
        )
        rgb_images.append(normalize_stack(stack))
 
    # 5. CoAtNet features
    features = extract_coatnet_features(rgb_images, batch_size=batch_size)
 
    assert features.shape == (n_slices, 768), \
        f"Unexpected shape: {features.shape}"
 
    print(f"Series features: {features.shape}")
 
    return features

In [ ]:
def extract_dino_series_features(series_path, n_slices=9, batch_size=9):
    """
    Read one DICOM series, select n_slices,
    and extract DINOv2 features in batches.

    Returns:
        Tensor [n_slices, 768]
    """

    # ---------------------------------------------
    # 1. Get sorted DICOM files
    # ---------------------------------------------

    sorted_files = get_sorted_dicom_files(series_path)

    if len(sorted_files) == 0:
        raise ValueError(
            f"No DICOM files found in {series_path}"
        )

    print(f"Found {len(sorted_files)} slices")

    # ---------------------------------------------
    # 2. Select slices
    # ---------------------------------------------

    selected_files = select_9_slices(
        sorted_files,
        n_slices
    )

    print(
        f"Selected {len(selected_files)} slices"
    )

    # ---------------------------------------------
    # 3. Read and normalize images
    # ---------------------------------------------

    images = []

    for i, filepath in enumerate(selected_files):

        ds = pydicom.dcmread(filepath)

        image = ds.pixel_array

        # uint8 [0,255]
        image = normalize_image(image)

        images.append(image)

    # ---------------------------------------------
    # 4. Batched DINOv2
    # ---------------------------------------------

    features = extract_dinov2_features(
        images,
        batch_size=batch_size
    )

    # ---------------------------------------------
    # 5. Verify shape
    # ---------------------------------------------

    assert features.shape == (
        n_slices,
        768
    ), f"Unexpected shape: {features.shape}"

    print(
        f"Series features: {features.shape}"
    )

    return features

In [ ]:
def extract_coatnet_study_features(row):
    
    features = torch.zeros(
        6, N_SLICES, 768,
        dtype=torch.float32
    )

    slice_mask = torch.zeros(
        6, N_SLICES,
        dtype=torch.float32
    )

    slot_mask = torch.zeros(
        6,
        dtype=torch.float32
    )

    study_id = row["StudyInstanceUID"]

    for slot_idx, column in enumerate(SLOT_COLUMNS):

        series_uid = row[column]

        # -----------------------------------------
        # Missing series
        # -----------------------------------------

        if pd.isna(series_uid) or str(series_uid).strip() == "":
            continue

        series_path = os.path.join(
            TEST_SERIES_ROOT,
            study_id,
            str(series_uid)
        )

        # -----------------------------------------
        # Existing series
        # -----------------------------------------

        series_features = extract_coatnet_series_features(
            series_path,
            n_slices=N_SLICES,
            batch_size=N_SLICES
        )

        # [9,768] -> [slot,9,768]
        features[slot_idx] = series_features

        # This slot exists
        slot_mask[slot_idx] = 1

        # These 9 slices exist
        slice_mask[slot_idx] = 1

    return (
        features,
        slice_mask,
        slot_mask
    )   

In [ ]:
def extract_dino_study_features(row):
    
    features = torch.zeros(
        6, 9, 768,
        dtype=torch.float32
    )

    slice_mask = torch.zeros(
        6, 9,
        dtype=torch.float32
    )

    slot_mask = torch.zeros(
        6,
        dtype=torch.float32
    )

    study_id = row["StudyInstanceUID"]

    for slot_idx, column in enumerate(SLOT_COLUMNS):

        series_uid = row[column]

        # -----------------------------------------
        # Missing series
        # -----------------------------------------

        if pd.isna(series_uid) or str(series_uid).strip() == "":
            continue

        series_path = os.path.join(
            TEST_SERIES_ROOT,
            study_id,
            str(series_uid)
        )

        # -----------------------------------------
        # Existing series
        # -----------------------------------------

        series_features = extract_dino_series_features(
            series_path,
            n_slices=9,
            batch_size=9
        )

        # [9,768] -> [slot,9,768]
        features[slot_idx] = series_features

        # This slot exists
        slot_mask[slot_idx] = 1

        # These 9 slices exist
        slice_mask[slot_idx] = 1

    return (
        features,
        slice_mask,
        slot_mask
    )

In [ ]:
OUTPUT_DIR = "/kaggle/working/test_features"

### Convert study series map dataframe

In [ ]:
import pandas as pd

INPUT_CSV = "/kaggle/input/competitions/rsna-knee-abnormality-detection/test_series.csv"
OUTPUT_CSV = "/kaggle/working/test_series_map.csv"

df = pd.read_csv(
    INPUT_CSV,
    dtype={
        "StudyInstanceUID": str,
        "SeriesInstanceUID": str
    }
)

SLOT_COLUMNS = [
    "Axial_F0", "Axial_F1",
    "Coronal_F0", "Coronal_F1",
    "Sagittal_F0", "Sagittal_F1"
]

SLOT_LOOKUP = {
    ("Axial", 0): "Axial_F0",
    ("Axial", 1): "Axial_F1",
    ("Coronal", 0): "Coronal_F0",
    ("Coronal", 1): "Coronal_F1",
    ("Sagittal", 0): "Sagittal_F0",
    ("Sagittal", 1): "Sagittal_F1"
}

study_ids = df["StudyInstanceUID"].drop_duplicates().tolist()
rows = []

for study_id in study_ids:
    row = {"StudyInstanceUID": study_id}

    # Initialize missing slots as pandas NA
    for slot in SLOT_COLUMNS:
        row[slot] = pd.NA

    study_df = df[df["StudyInstanceUID"] == study_id]

    for _, series in study_df.iterrows():
        plane = series["Anatomical_Plane"]
        fluid = int(series["Fluid_Sensitive"])

        slot = SLOT_LOOKUP.get((plane, fluid))

        if slot is not None and pd.isna(row[slot]):
            row[slot] = series["SeriesInstanceUID"]

    rows.append(row)

test_series_map = pd.DataFrame(
    rows,
    columns=["StudyInstanceUID"] + SLOT_COLUMNS
)

test_series_map.to_csv(OUTPUT_CSV, index=False)

# Read back to verify missing values appear as NaN
test_series_map = pd.read_csv(OUTPUT_CSV)

print("Shape:", test_series_map.shape)
print("Saved to:", OUTPUT_CSV)
display(test_series_map)

In [ ]:

import os
import pandas as pd
import torch
import numpy as np

# ---------------------------------------------
# 1. Configuration
# ---------------------------------------------
df = pd.read_csv("/kaggle/working/test_series_map.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

SAMPLE_PATH = (
    "/kaggle/input/competitions/"
    "rsna-knee-abnormality-detection/sample_submission.csv"
)
SUBMISSION_PATH = "/kaggle/working/submission.csv"

LABEL_COLS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

# Check test study IDs
assert "StudyInstanceUID" in df.columns
assert df["StudyInstanceUID"].is_unique, "Duplicate study IDs in test map!"

# Set classifiers to evaluation mode
dino_model.eval()
coatnet_model.eval()

results = []

# ---------------------------------------------
# 2. Inference
# ---------------------------------------------
for idx, row in df.iterrows():
    study_id = str(row["StudyInstanceUID"])
    output_path = os.path.join(OUTPUT_DIR, f"{study_id}.pt")

    try:
        print(f"\n[{idx + 1}/{len(df)}] Processing {study_id}")

        # Extract DINOv2 features
        dino_features, dino_slice_mask, dino_slot_mask = (
            extract_dino_study_features(row)
        )

        # Extract CoAtNet features
        coatnet_features, coatnet_slice_mask, coatnet_slot_mask = (
            extract_coatnet_study_features(row)
        )

        # Convert to tensors and add batch dimension
        dino_features = torch.as_tensor(
            dino_features, dtype=torch.float32
        ).unsqueeze(0)
        dino_slice_mask = torch.as_tensor(
            dino_slice_mask, dtype=torch.float32
        ).unsqueeze(0)
        dino_slot_mask = torch.as_tensor(
            dino_slot_mask, dtype=torch.float32
        ).unsqueeze(0)

        coatnet_features = torch.as_tensor(
            coatnet_features, dtype=torch.float32
        ).unsqueeze(0)
        coatnet_slice_mask = torch.as_tensor(
            coatnet_slice_mask, dtype=torch.float32
        ).unsqueeze(0)
        coatnet_slot_mask = torch.as_tensor(
            coatnet_slot_mask, dtype=torch.float32
        ).unsqueeze(0)

        # Get each model's device
        dino_device = next(dino_model.parameters()).device
        coatnet_device = next(coatnet_model.parameters()).device

        # Predict
        with torch.inference_mode():
            dino_logits, _, dino_slot_weights = dino_model(
                dino_features.to(dino_device),
                dino_slice_mask.to(dino_device),
                dino_slot_mask.to(dino_device)
            )

            coatnet_logits, _, coatnet_slot_weights = coatnet_model(
                coatnet_features.to(coatnet_device),
                coatnet_slice_mask.to(coatnet_device),
                coatnet_slot_mask.to(coatnet_device)
            )

            dino_probs = torch.sigmoid(dino_logits).cpu()
            coatnet_probs = torch.sigmoid(coatnet_logits).cpu()

            # 30% DINOv2 + 70% CoAtNet
            ensemble_probs = (
                0.3 * dino_probs + 0.7 * coatnet_probs
            )

        probabilities = ensemble_probs.squeeze(0).numpy()

        # Validate this study's predictions
        if probabilities.shape != (len(LABEL_COLS),):
            raise ValueError(
                f"Expected {len(LABEL_COLS)} predictions, "
                f"got shape {probabilities.shape}"
            )

        if not np.isfinite(probabilities).all():
            raise ValueError("Predictions contain NaN or infinity")

        if ((probabilities < 0) | (probabilities > 1)).any():
            raise ValueError("Predictions must be between 0 and 1")

        # Save features, masks, and predictions for this study
        torch.save(
            {
                "study_id": study_id,
                "dino_features": dino_features.squeeze(0).cpu(),
                "dino_slice_mask": dino_slice_mask.squeeze(0).cpu(),
                "dino_slot_mask": dino_slot_mask.squeeze(0).cpu(),
                "coatnet_features": coatnet_features.squeeze(0).cpu(),
                "coatnet_slice_mask": coatnet_slice_mask.squeeze(0).cpu(),
                "coatnet_slot_mask": coatnet_slot_mask.squeeze(0).cpu(),
                "dino_probabilities": dino_probs.squeeze(0),
                "coatnet_probabilities": coatnet_probs.squeeze(0),
                "ensemble_probabilities": ensemble_probs.squeeze(0),
            },
            output_path
        )

        # Store results in memory; no intermediate CSV
        result = {"StudyInstanceUID": study_id}
        result.update(dict(zip(LABEL_COLS, probabilities.tolist())))
        results.append(result)

        print("Ensemble probabilities:", probabilities)

    except Exception as e:
        print(f"ERROR processing study {study_id}: {e}")
        raise

# ---------------------------------------------
# 3. Create submission.csv directly
# ---------------------------------------------
if len(results) != len(df):
    raise RuntimeError(
        f"Expected {len(df)} predictions, got {len(results)}"
    )

predictions_df = pd.DataFrame(results)

sample_submission = pd.read_csv(SAMPLE_PATH)

# Confirm expected competition columns
expected_columns = sample_submission.columns.tolist()

if set(expected_columns) != set(
    ["StudyInstanceUID"] + LABEL_COLS
):
    raise ValueError(
        "Sample submission columns do not match expected label columns. "
        f"Found: {expected_columns}"
    )

# Align predictions to the sample submission's study order
submission_df = sample_submission[["StudyInstanceUID"]].merge(
    predictions_df,
    on="StudyInstanceUID",
    how="left",
    validate="one_to_one"
)

# Restore exact required column order
submission_df = submission_df[expected_columns]

# Final validation
if submission_df.shape != sample_submission.shape:
    raise ValueError(
        f"Submission shape {submission_df.shape} does not match "
        f"sample shape {sample_submission.shape}"
    )

if submission_df.isna().any().any():
    raise ValueError("Submission contains missing values")

prediction_values = submission_df[LABEL_COLS].to_numpy(dtype=np.float64)

if not np.isfinite(prediction_values).all():
    raise ValueError("Submission contains NaN or infinite values")

if ((prediction_values < 0) | (prediction_values > 1)).any():
    raise ValueError("Submission probabilities must be between 0 and 1")

# Save the only prediction CSV
submission_df.to_csv(SUBMISSION_PATH, index=False)

print("\nSubmission file created successfully!")
print("Saved to:", SUBMISSION_PATH)
print("Shape:", submission_df.shape)
display(submission_df)
